# 82 Veeva Link T3 features at NPI x cohort (Phase 2, R1)

Task R1 of `80_veeva_link_phase2_working_plan.md`. Builds the 16 T3 columns of `56_` (best decile, `CLINICAL_LEADER` disease / class deciles, `FOCUS_AREA`, `DIGITAL_REACH`, `INDEX`) at the NBA 3.1 grain `NPI_NUMBER` x `COHORT_ID` for all 43 cohorts of `71_`, in two versions (P2-D2):

| Version | Scores used | Note |
| --- | --- | --- |
| `snapshot` | frozen snapshot `52_` (`IS_LATEST_FLAG = 'Y'`) for every cohort | same as `56_`; includes the 'stale latest' rows found in `81_` |
| `asof` | the load picked by `81_` for the cohort (latest load on or before the cut-off) | null for cohorts without a pick in all four score tables (cut-off before 2024-08-01) |

T3 stays in its own table (P2-D1); nothing here is joined into `features_phase1/t1_t2`. Missing rule: all T3 null for `NOT_IN_VEEVA`; deciles null (never 0) for unscored experts; `veeva_dr_any_inscope` 0 for matched experts without a DR score.

Checks L1 to L4 and L6 (section 5.2 of `80_`) written to `OUT/features_phase2/checks/run=<ts>/`.

In [0]:
from pyspark.sql import functions as F
import pandas as pd, datetime as dt, json, time

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
SNAP, CFG = f"{OUT}snapshot/load=2025-06-30/", f"{OUT}config/"
P2 = f"{OUT}features_phase2/"
SH = P2 + "score_history/"
RUN_TS = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")
SCORE_TABLES = ["clinical_leader", "focus_area", "digital_reach_focus_area", "index"]

TH = {"CKD": "ckd", "Heart failure": "hf", "T2D": "t2d", "T1D": "t1d", "Hypertension": "htn", "Kerendia / MRA": "mra_class", "SGLT2 inhibitors": "sglt2_class"}
IDX = {"Publication Index": "veeva_index_publication", "Clinical Trial Index": "veeva_index_trial", "Event Index": "veeva_index_event",
       "Guideline Index": "veeva_index_guideline", "Scientific Reach": "veeva_scientific_reach"}
T3_COLS = ["veeva_best_decile", "veeva_decile_bin"] + [f"veeva_cl_decile_{v}" for v in TH.values()] + \
          ["veeva_fa_decile_max_inscope", "veeva_dr_any_inscope"] + list(IDX.values())
KEY_COLS = ["NPI_NUMBER", "PROVIDER_ID", "COHORT_ID", "t3_version", "veeva_coverage_status"]
LOG_COLS = ["veeva_score_load_date", "veeva_score_age_months"]

# NPI grain exactly as in 71_ (one row per NPI; 11 NPIs on 2+ providers kept once): take keys from the 71_ output
spine = spark.read.parquet(f"{OUT}spine/veeva_npi_se_map").select("PROVIDER_ID", "SE_ID", "IN_VEEVA")
base = (spark.read.parquet(f"{OUT}features_phase1/t1_t2").filter("COHORT_ID = '2025-02_offset7'")
             .select("NPI_NUMBER", "PROVIDER_ID", "veeva_coverage_status").join(spine, "PROVIDER_ID", "left"))
matched = spine.filter("IN_VEEVA = 1").select("SE_ID").distinct()
cohorts = spark.read.parquet(f"{OUT}features_phase1/cohort_windows").select("COHORT_ID", "cohort_set", "cohort_cutoff").toPandas()
cohorts["cohort_cutoff"] = pd.to_datetime(cohorts.cohort_cutoff).dt.date
pick = spark.read.parquet(SH + "cohort_load_pick").toPandas()
fd = spark.read.parquet(CFG + "focus_dictionary_v1").toPandas()
print("NPI rows:", base.count(), "| matched experts:", matched.count(), "| cohorts:", len(cohorts), "| T3 columns:", len(T3_COLS))

In [0]:
# Every score source becomes rows (SE_ID, family, theme, SCORE): family in CL / FA / DR / IDX; theme = TH key for CL, index type for IDX
cl_k = spark.createDataFrame(fd[fd.source_table == "clinical_leader"][["focus_name", "cl_type", "theme"]]
                             .rename(columns={"focus_name": "FOCUS_NAME", "cl_type": "TYPE"}))
fa_k = spark.createDataFrame(fd[fd.source_table == "focus_area"][["focus_name", "theme"]].rename(columns={"focus_name": "FOCUS_NAME"}))
dr_k = spark.createDataFrame(fd[fd.source_table == "digital_reach_focus_area"][["focus_name", "theme"]].rename(columns={"focus_name": "FOCUS_NAME"}))

def shape(t, df, score):
    """df has SE_ID, FOCUS_NAME / TYPE and a score column; returns SE_ID, family, theme, SCORE (in-scope, valid range only)"""
    if t == "index":
        return (df.filter(F.col("TYPE").isin(list(IDX))).select("SE_ID", F.lit("IDX").alias("family"), F.col("TYPE").alias("theme"),
                                                              F.col(score).cast("double").alias("SCORE")))
    k, fam, keys = {"clinical_leader": (cl_k, "CL", ["FOCUS_NAME", "TYPE"]), "focus_area": (fa_k, "FA", ["FOCUS_NAME"]),
                    "digital_reach_focus_area": (dr_k, "DR", ["FOCUS_NAME"])}[t]
    return (df.join(F.broadcast(k), keys).select("SE_ID", F.lit(fam).alias("family"), "theme", F.col(score).cast("double").alias("SCORE"))
              .filter("SCORE BETWEEN 1 AND 10"))

SNAP_SCORE = {"clinical_leader": "SE_DECILE", "focus_area": "FOCUS_DECILE", "digital_reach_focus_area": "SE_DECILE", "index": "VALUE"}
snap_rows = None
for t in SCORE_TABLES:
    r = shape(t, spark.read.parquet(SNAP + t).join(matched, "SE_ID"), SNAP_SCORE[t])
    snap_rows = r if snap_rows is None else snap_rows.unionByName(r)

arch = {t: spark.read.parquet(f"{SH}loads/{t}").withColumn("LOAD_ID", F.col("LOAD_ID").cast("string")) for t in SCORE_TABLES}
def load_rows(loads):
    """loads: {table: LOAD_ID} -> score rows of those loads"""
    u = None
    for t, l in loads.items():
        r = shape(t, arch[t].filter(F.col("LOAD_ID") == l), "SCORE")
        u = r if u is None else u.unionByName(r)
    return u

display(snap_rows.groupBy("family").agg(F.count("*").alias("rows"), F.countDistinct("SE_ID").alias("experts")).orderBy("family"))

In [0]:
def t3_per_se(rows):
    dec = rows.filter(F.col("family").isin("CL", "FA", "DR"))
    best = dec.groupBy("SE_ID").agg(F.max("SCORE").cast("int").alias("veeva_best_decile"))
    cl = (dec.filter((F.col("family") == "CL") & F.col("theme").isin(list(TH))).groupBy("SE_ID").pivot("theme", list(TH))
             .agg(F.max("SCORE").cast("int")))
    for k, v in TH.items():
        cl = cl.withColumnRenamed(k, f"veeva_cl_decile_{v}")
    fa = dec.filter("family = 'FA'").groupBy("SE_ID").agg(F.max("SCORE").cast("int").alias("veeva_fa_decile_max_inscope"))
    dr = dec.filter("family = 'DR'").select("SE_ID").distinct().withColumn("veeva_dr_any_inscope", F.lit(1))
    ix = rows.filter("family = 'IDX'").groupBy("SE_ID").pivot("theme", list(IDX)).agg(F.max("SCORE"))
    for k, v in IDX.items():
        ix = ix.withColumnRenamed(k, v)
    return best.join(cl, "SE_ID", "full").join(fa, "SE_ID", "full").join(dr, "SE_ID", "full").join(ix, "SE_ID", "full")

def to_npi(t3se, cohort_ids, version, load_date):
    """Attach to the NPI grain, apply the missing rule, repeat for the given cohorts"""
    d = base.join(t3se, "SE_ID", "left")
    inv = F.col("IN_VEEVA") == 1
    d = (d.withColumn("veeva_decile_bin", F.when(~inv, None).when(F.col("veeva_best_decile").isNull(), "unscored")
                      .when(F.col("veeva_best_decile") >= 8, "8-10").when(F.col("veeva_best_decile") >= 4, "4-7").otherwise("1-3"))
          .withColumn("veeva_dr_any_inscope", F.when(inv, F.coalesce("veeva_dr_any_inscope", F.lit(0))).cast("int")))
    for c in T3_COLS:   # nothing for NOT_IN_VEEVA (principle 2)
        d = d.withColumn(c, F.when(inv, F.col(c)))
    cw = spark.createDataFrame(cohorts[cohorts.COHORT_ID.isin(cohort_ids)][["COHORT_ID", "cohort_cutoff"]].astype({"cohort_cutoff": str}))
    d = (d.crossJoin(F.broadcast(cw)).withColumn("t3_version", F.lit(version))
          .withColumn("veeva_score_load_date", F.lit(load_date).cast("date"))
          .withColumn("veeva_score_age_months", F.when(F.lit(load_date).isNotNull(),
                      F.floor(F.months_between(F.col("cohort_cutoff").cast("date"), F.lit(load_date).cast("date"))).cast("int"))))
    return d.select(*KEY_COLS, *T3_COLS, *LOG_COLS)

In [0]:
t0 = time.time()
ALL = sorted(cohorts.COHORT_ID)

# snapshot: one T3 for every cohort; load date = 2025-03-06 (child-table cut-off of the snapshot manifest)
snap_t3 = to_npi(t3_per_se(snap_rows), ALL, "snapshot", "2025-03-06")
snap_t3.write.mode("overwrite").partitionBy("COHORT_ID").parquet(P2 + "t3/version=snapshot")

# as-of: one T3 per distinct set of picked loads; cohorts without a pick in all 4 tables get an all-null T3
pk = pick.pivot_table(index="COHORT_ID", columns="table", values="LOAD_ID", aggfunc="first").reindex(ALL)
pk_date = pick.pivot_table(index="COHORT_ID", columns="table", values="LOAD_DATE", aggfunc="first").reindex(ALL)
full = pk[SCORE_TABLES].notna().all(axis=1)
groups = pk[full].groupby(SCORE_TABLES).groups   # {(cl, fa, dr, idx): [cohort ids]}
parts = []
for loads, cids in groups.items():
    ld = str(max(pk_date.loc[list(cids)[0], SCORE_TABLES]))   # newest of the 4 picked load dates
    parts.append(to_npi(t3_per_se(load_rows(dict(zip(SCORE_TABLES, loads)))), list(cids), "asof", ld))
none_ids = list(pk[~full].index)
empty_se = spark.createDataFrame([], "SE_ID string")
for c in T3_COLS:
    empty_se = empty_se.withColumn(c, F.lit(None).cast(snap_t3.schema[c].dataType))
parts.append(to_npi(empty_se.drop("veeva_decile_bin"), none_ids, "asof", None)
             .withColumn("veeva_decile_bin", F.lit(None).cast("string")).select(*KEY_COLS, *T3_COLS, *LOG_COLS))
asof_t3 = parts[0]
for p in parts[1:]:
    asof_t3 = asof_t3.unionByName(p)
asof_t3.write.mode("overwrite").partitionBy("COHORT_ID").parquet(P2 + "t3/version=asof")
print(f"as-of load sets: {len(groups)} | cohorts with as-of T3: {int(full.sum())} | without: {len(none_ids)} | minutes: {(time.time() - t0) / 60:.1f}")

In [0]:
t3 = spark.read.parquet(P2 + "t3")   # version and COHORT_ID come back as partition columns
t3 = t3.withColumnRenamed("version", "t3_version_part")
rows = []
# L1 grain: one row per NPI x cohort x version, NPI set = 71_
g = t3.groupBy("t3_version_part", "COHORT_ID").agg(F.count("*").alias("n"), F.countDistinct("NPI_NUMBER").alias("d")).toPandas()
n71 = base.count()
l1 = bool(((g.n == n71) & (g.d == n71)).all() and len(g) == 2 * len(cohorts))
rows.append({"check": "L1", "name": "one row per NPI x cohort x version", "value": f"{g.n.min()}..{g.n.max()} rows, {len(g)} partitions",
             "expected": f"{n71} rows x {2 * len(cohorts)}", "pass": l1})
# L2 snapshot C2 = 56_ t3 anchor 2025-10-23 on every T3 column (56_ is keyed on PROVIDER_ID)
ref = spark.read.parquet(f"{OUT}features/t3/anchor=2025-10-23").select("PROVIDER_ID", *[F.col(c).alias(f"r_{c}") for c in T3_COLS])
c2 = t3.filter("t3_version_part = 'snapshot' AND COHORT_ID = '2025-02_offset7'").join(ref, "PROVIDER_ID", "left")
mism = c2.select(*[F.sum((~F.col(c).eqNullSafe(F.col(f"r_{c}"))).cast("int")).alias(c) for c in T3_COLS]).toPandas().T[0]
rows.append({"check": "L2", "name": "snapshot C2 = 56_ anchor 2025-10-23", "value": f"{int(mism.sum())} mismatching values in {len(T3_COLS)} columns",
             "expected": "0", "pass": bool(mism.sum() == 0)})
sc = c2.filter("veeva_best_decile IS NOT NULL").count()
rows.append({"check": "L2", "name": "snapshot C2 scored NPIs (57_: 147,487 providers less NPI rule)", "value": str(sc),
             "expected": "147487 less no-NPI / duplicate NPIs", "pass": abs(sc - 147487) <= 200})
# L3 missing is not zero
nv = t3.filter("veeva_coverage_status = 'NOT_IN_VEEVA'").select(sum(F.col(c).isNotNull().cast("int") for c in T3_COLS).alias("x")).agg(F.sum("x")).first()[0]
z = t3.select(sum((F.col(c) == 0).cast("int") for c in T3_COLS if "decile" in c and c != "veeva_decile_bin").alias("x")).agg(F.sum("x")).first()[0]
rows.append({"check": "L3", "name": "non-null T3 for NOT_IN_VEEVA / deciles = 0", "value": f"{nv} / {z}", "expected": "0 / 0", "pass": nv == 0 and z == 0})
# L4 as-of is as-of
lo = (t3.filter("t3_version_part = 'asof'").select("COHORT_ID", "veeva_score_load_date").distinct().toPandas()
        .merge(cohorts, on="COHORT_ID"))
bad = lo[lo.veeva_score_load_date.notna() & (pd.to_datetime(lo.veeva_score_load_date).dt.date > lo.cohort_cutoff)]
null_ok = lo[lo.cohort_cutoff < dt.date(2024, 8, 1)].veeva_score_load_date.isna().all()
rows.append({"check": "L4", "name": "as-of load date <= cut-off; null before first CL load", "value": f"{len(bad)} cohorts over; nulls ok: {null_ok}",
             "expected": "0; True", "pass": len(bad) == 0 and bool(null_ok)})
# L6 fixed schema: columns vs lock (first run writes the lock)
cols = [f.name for f in spark.read.parquet(P2 + "t3/version=snapshot").schema.fields]
lock_p = P2 + "schema_lock/t3_columns.json"
try:
    prev = json.loads(dbutils.fs.head(lock_p))
    same, note = prev == cols, "compared with previous run"
except Exception:
    dbutils.fs.put(lock_p, json.dumps(cols), overwrite=True)
    same, note = True, "first run: lock written"
rows.append({"check": "L6", "name": "columns = schema lock", "value": str(len(cols)), "expected": str(len(cols)), "pass": bool(same), "note": note})
chk = pd.DataFrame(rows).fillna("")
chk["run_ts"] = RUN_TS
spark.createDataFrame(chk.astype(str)).write.mode("overwrite").parquet(f"{P2}checks/run={RUN_TS}")
display(chk)
if mism.sum():
    display(mism[mism > 0].rename("mismatches").reset_index())
print("ALL CHECKS PASS:", bool(chk["pass"].astype(bool).all()), "| run:", RUN_TS)

In [0]:
a = t3.filter("t3_version_part = 'asof' AND veeva_score_load_date IS NOT NULL")
s = t3.filter("t3_version_part = 'snapshot'").select("NPI_NUMBER", "COHORT_ID", *[F.col(c).alias(f"s_{c}") for c in T3_COLS])
x = a.join(s, ["NPI_NUMBER", "COHORT_ID"]).filter("veeva_coverage_status != 'NOT_IN_VEEVA'")
cmp_cols = ["veeva_best_decile", "veeva_decile_bin", "veeva_cl_decile_ckd", "veeva_fa_decile_max_inscope", "veeva_dr_any_inscope", "veeva_index_publication"]
diff = (x.groupBy("COHORT_ID", "veeva_score_load_date").agg(F.count("*").alias("matched_npis"),
          *[F.round(F.avg((~F.col(c).eqNullSafe(F.col(f"s_{c}"))).cast("int")), 4).alias(f"share_diff_{c[6:]}") for c in cmp_cols])
         .orderBy("COHORT_ID").toPandas())
spark.createDataFrame(diff.astype({"veeva_score_load_date": str})).write.mode("overwrite").parquet(f"{P2}t3_version_diff")
display(diff)

## R5 Cross features (practice and IDN experts)

Logic of `60_`: practice (L1) = BEST_AFFL account with `BRAND_RANK = 1`; IDN (L4) = `L4_IDN_BHO_ID` of `mdm_stg_acct_hrchy_ker`, with org name null / 'NA' / 'Multiple Doctors' = no IDN. Groups are formed over the 294,542 NPIs of `71_`; every HCP is removed from its own group. Expert = `veeva_best_decile` 8 to 10 from the T3 version of the same cohort (P2-D6), so the cross table has the same two versions as T3.

| Column | Rule |
| --- | --- |
| `veeva_idn_expert_share` | experts / HCPs in the IDN, self excluded; null if no IDN or no other HCP in it |
| `veeva_idn_expert_share_zero_flag` | 1 if the share is 0, else 0; null where the share is null |
| `veeva_practice_expert_peer_flag` | 1 if at least one other HCP in the practice is an expert; null if no practice or no other HCP in it. Use only together with practice size (R2-5) |
| `veeva_practice_n_peers`, `veeva_idn_n_peers` | logged group sizes (self excluded), not candidate features |

The groups are today's affiliation for every cohort (same as the colocation block). In `asof`, cohorts without an as-of T3 get null cross features.

In [0]:
C = "`ph-com-ai-us-prod-catalog-2553213839599380`"
xw = spark.table(f"{C}.kerendia_nba_3.source_provider_bhid_npi_mapping").select("PROVIDER_ID", "BH_ID")
affl = (spark.table(f"{C}.pace_ker_in_house_schema.cust_mdm_best_affl_publish_pd_outp_static")
             .filter("BRAND_RANK = 1").select("BH_ID", F.col("BHO_ID").alias("L1_ID")))
hier = (spark.table(f"{C}.pace_ker_in_house_schema.mdm_stg_acct_hrchy_ker")
             .select(F.col("L1_CHILD_BHO_ID").alias("L1_ID"), "L4_IDN_BHO_ID", "L4_IDN_ORGNAME")
             .withColumn("L4_ID", F.when(F.col("L4_IDN_ORGNAME").isNull() | F.upper(F.trim("L4_IDN_ORGNAME")).isin("NA", "MULTIPLE DOCTORS"), None)
                                   .otherwise(F.col("L4_IDN_BHO_ID")))
             .select("L1_ID", "L4_ID"))
# checked 2026-11-23: 1 BH_ID per provider, 1 L1 per BH_ID, 1 row per L1 in the hierarchy -> no fan-out
grp = (base.select("NPI_NUMBER", "PROVIDER_ID").join(xw, "PROVIDER_ID", "left").join(affl, "BH_ID", "left")
           .join(hier, "L1_ID", "left").select("NPI_NUMBER", "L1_ID", "L4_ID"))
g = grp.agg(F.count("*").alias("rows"), F.countDistinct("NPI_NUMBER").alias("npis"), F.count("L1_ID").alias("with_practice"),
            F.count("L4_ID").alias("in_idn"), F.countDistinct("L1_ID").alias("practices"), F.countDistinct("L4_ID").alias("idns")).toPandas()
display(g)

In [0]:
CROSS_COLS = ["veeva_idn_expert_share", "veeva_idn_expert_share_zero_flag", "veeva_practice_expert_peer_flag"]
CROSS_LOG = ["veeva_practice_n_peers", "veeva_idn_n_peers"]
t0 = time.time()
t3v = (spark.read.parquet(P2 + "t3").withColumnRenamed("version", "t3_version_part")
            .select("NPI_NUMBER", "COHORT_ID", F.col("t3_version_part").alias("t3_version"), "veeva_best_decile", "veeva_score_load_date"))
# expert 1/0; null when the version has no T3 for that cohort (as-of before 2024-08-01)
has_t3 = (F.col("t3_version") == "snapshot") | F.col("veeva_score_load_date").isNotNull()
d = (t3v.join(grp, "NPI_NUMBER")
        .withColumn("EXPERT", F.when(has_t3, (F.coalesce("veeva_best_decile", F.lit(0)) >= 8).cast("int"))))
k = ["t3_version", "COHORT_ID"]
l1 = d.filter("L1_ID IS NOT NULL").groupBy(*k, "L1_ID").agg(F.count("*").alias("N1"), F.sum("EXPERT").alias("E1"))
l4 = d.filter("L4_ID IS NOT NULL").groupBy(*k, "L4_ID").agg(F.count("*").alias("N4"), F.sum("EXPERT").alias("E4"))
x = d.join(l1, k + ["L1_ID"], "left").join(l4, k + ["L4_ID"], "left")
own = F.coalesce("EXPERT", F.lit(0))
x = (x.withColumn("veeva_practice_n_peers", (F.col("N1") - 1).cast("int"))
      .withColumn("veeva_idn_n_peers", (F.col("N4") - 1).cast("int"))
      .withColumn("pe1", F.col("E1") - own).withColumn("pe4", F.col("E4") - own))   # self excluded
ok = F.col("EXPERT").isNotNull()
x = (x.withColumn("veeva_idn_expert_share", F.when(ok & (F.col("veeva_idn_n_peers") > 0), F.col("pe4") / F.col("veeva_idn_n_peers")))
      .withColumn("veeva_idn_expert_share_zero_flag", F.when(F.col("veeva_idn_expert_share").isNotNull(), (F.col("veeva_idn_expert_share") == 0).cast("int")))
      .withColumn("veeva_practice_expert_peer_flag", F.when(ok & (F.col("veeva_practice_n_peers") > 0), (F.col("pe1") > 0).cast("int")))
      .select("NPI_NUMBER", "COHORT_ID", "t3_version", *CROSS_COLS, *CROSS_LOG))
x.write.mode("overwrite").partitionBy("t3_version", "COHORT_ID").parquet(P2 + "cross_tmp")
# final layout cross/version=<v>/COHORT_ID=<id>/ like T3
for v in ["snapshot", "asof"]:
    spark.read.parquet(f"{P2}cross_tmp/t3_version={v}").withColumn("t3_version", F.lit(v)) \
         .select("NPI_NUMBER", "COHORT_ID", "t3_version", *CROSS_COLS, *CROSS_LOG) \
         .write.mode("overwrite").partitionBy("COHORT_ID").parquet(f"{P2}cross/version={v}")
dbutils.fs.rm(P2 + "cross_tmp", True)
print(f"minutes: {(time.time() - t0) / 60:.1f}")
cr = spark.read.parquet(P2 + "cross")
display(cr.filter("COHORT_ID = '2025-02_offset7'").groupBy("version").agg(
    F.count("*").alias("rows"), *[F.count(c).alias(f"nn_{c[6:]}") for c in CROSS_COLS],
    F.round(F.avg("veeva_idn_expert_share"), 4).alias("mean_idn_share"), F.sum("veeva_idn_expert_share_zero_flag").alias("idn_zero"),
    F.sum("veeva_practice_expert_peer_flag").alias("practice_flag_1")))

In [0]:
rows = []
cr = spark.read.parquet(P2 + "cross")
g = cr.groupBy("version", "COHORT_ID").agg(F.count("*").alias("n"), F.countDistinct("NPI_NUMBER").alias("d")).toPandas()
rows.append({"check": "L1", "name": "cross: one row per NPI x cohort x version", "value": f"{g.n.min()}..{g.n.max()} rows, {len(g)} partitions",
             "expected": f"{n71} rows x {2 * len(cohorts)}", "pass": bool(((g.n == n71) & (g.d == n71)).all() and len(g) == 2 * len(cohorts))})

# L5 self excluded: recount 100 random HCPs (snapshot, C2) by a direct self-join on the group keys, excluding the HCP itself
C2 = "2025-02_offset7"
exp = (spark.read.parquet(P2 + "t3/version=snapshot").filter(f"COHORT_ID = '{C2}'")
            .select("NPI_NUMBER", (F.coalesce("veeva_best_decile", F.lit(0)) >= 8).cast("int").alias("EXPERT")).join(grp, "NPI_NUMBER"))
sample = exp.filter("L1_ID IS NOT NULL OR L4_ID IS NOT NULL").orderBy(F.rand(42)).limit(100).select("NPI_NUMBER", "L1_ID", "L4_ID")
peers = exp.select(F.col("NPI_NUMBER").alias("P_NPI"), F.col("L1_ID").alias("P_L1"), F.col("L4_ID").alias("P_L4"), F.col("EXPERT").alias("P_EXP"))
sj = sample.join(peers, (F.col("P_NPI") != F.col("NPI_NUMBER")) & ((F.col("P_L1") == F.col("L1_ID")) | (F.col("P_L4") == F.col("L4_ID"))), "left")
rc = (sj.groupBy("NPI_NUMBER").agg(
        F.sum(F.when(F.col("P_L1") == F.col("L1_ID"), 1).otherwise(0)).alias("n1"), F.sum(F.when(F.col("P_L1") == F.col("L1_ID"), F.col("P_EXP")).otherwise(0)).alias("e1"),
        F.sum(F.when(F.col("P_L4") == F.col("L4_ID"), 1).otherwise(0)).alias("n4"), F.sum(F.when(F.col("P_L4") == F.col("L4_ID"), F.col("P_EXP")).otherwise(0)).alias("e4"))
        .join(sample, "NPI_NUMBER")
        .withColumn("r_share", F.when(F.col("L4_ID").isNotNull() & (F.col("n4") > 0), F.col("e4") / F.col("n4")))
        .withColumn("r_flag", F.when(F.col("L1_ID").isNotNull() & (F.col("n1") > 0), (F.col("e1") > 0).cast("int")))
        .withColumn("r_n1", F.when(F.col("L1_ID").isNotNull(), F.col("n1"))).withColumn("r_n4", F.when(F.col("L4_ID").isNotNull(), F.col("n4"))))
l5 = rc.join(cr.filter(f"version = 'snapshot' AND COHORT_ID = '{C2}'"), "NPI_NUMBER")
bad = l5.filter(~F.col("r_share").eqNullSafe(F.col("veeva_idn_expert_share")) | ~F.col("r_flag").eqNullSafe(F.col("veeva_practice_expert_peer_flag"))
                | ~F.col("r_n1").eqNullSafe(F.col("veeva_practice_n_peers")) | ~F.col("r_n4").eqNullSafe(F.col("veeva_idn_n_peers"))).count()
rows.append({"check": "L5", "name": "self excluded: 100 HCPs recounted by direct self-join", "value": f"{bad} mismatches of {l5.count()}", "expected": "0 of 100",
             "pass": bad == 0 and l5.count() == 100})
# L3-style: as-of cross null where as-of T3 is null
nn = cr.filter("version = 'asof'").join(spark.createDataFrame(pd.DataFrame({"COHORT_ID": none_ids})), "COHORT_ID") \
       .select(sum(F.col(c).isNotNull().cast("int") for c in CROSS_COLS).alias("x")).agg(F.sum("x")).first()[0] or 0
rows.append({"check": "L4", "name": "as-of cross null for cohorts without as-of T3", "value": str(nn), "expected": "0", "pass": nn == 0})
# L6 fixed schema
cols = [f.name for f in spark.read.parquet(P2 + "cross/version=snapshot").schema.fields]
lock_p = P2 + "schema_lock/cross_columns.json"
try:
    prev = json.loads(dbutils.fs.head(lock_p)); same, note = prev == cols, "compared with previous run"
except Exception:
    dbutils.fs.put(lock_p, json.dumps(cols), overwrite=True); same, note = True, "first run: lock written"
rows.append({"check": "L6", "name": "cross columns = schema lock", "value": str(len(cols)), "expected": str(len(cols)), "pass": bool(same), "note": note})
chk5 = pd.DataFrame(rows).fillna("")
chk5["run_ts"] = RUN_TS
spark.createDataFrame(chk5.astype(str)).write.mode("overwrite").parquet(f"{P2}checks/run={RUN_TS}_cross")
display(chk5)
print("ALL CROSS CHECKS PASS:", bool(chk5["pass"].astype(bool).all()), "| run:", RUN_TS)